In [1]:
import pandas as pd
import os
import re
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
import geopandas as gpd
from sklearn.impute import KNNImputer
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline

In [2]:
def clean_rainfall_excel(file_path):
    df = pd.read_excel(file_path, header=None)

    # --- Step 1: Extract District and Station ---
    district = station = None
    for i in range(5):
        for j in range(5):
            cell = str(df.iloc[i, j])
            if 'District' in cell:
                try:
                    district = cell.split(':')[1].strip()
                except IndexError:
                    district = 'Unknown'
            if 'Station' in cell:
                try:
                    station = cell.split(':')[1].strip()
                except IndexError:
                    station = 'Unknown'

    if not district:
        district =  station

    # --- Step 2: Loop through each year block ---
    all_rows = []
    i = 0
    while i < len(df):
        row = df.iloc[i]
        first_cell = str(row.iloc[0])
        # Match 4-digit year
        if re.match(r'^\d{4}$', first_cell.strip()):
            year = int(first_cell)
            months = df.iloc[i].iloc[1:13].tolist()
            weeks_block = df.iloc[i+1:i+5].reset_index(drop=True)

            for week_idx in range(4):
                week_label = weeks_block.iloc[week_idx, 0]
                for month_idx in range(12):
                    month = months[month_idx]
                    rainfall = weeks_block.iloc[week_idx, month_idx + 1]
                    all_rows.append({
                        'District': district,
                        'Station': station,
                        'Year': year,
                        'Month': month,
                        'Week': week_label,
                        'Rainfall': rainfall
                    })
            i += 5  # Move to next year block
        else:
            i += 1  # Skip irrelevant rows

    return pd.DataFrame(all_rows)


In [3]:
folder_path = '../RawDataset/rainfall'

all_data = []
for file in os.listdir(folder_path):
    if file.endswith('.xlsx'):
        file_path = os.path.join(folder_path, file)
        print(f"Processing: {file_path}")
        df = clean_rainfall_excel(file_path)
        all_data.append(df)

combined_df = pd.concat(all_data, ignore_index=True)
# Save to CSV
output_csv_path = '../RawDataset/rainfall/combined_rainfall_data.csv'
combined_df.to_csv(output_csv_path, index=False)

print(f"Saved combined data to {output_csv_path}")

Processing: ../RawDataset/rainfall/rain_unguja_magharibi_b.xlsx
Processing: ../RawDataset/rainfall/WETE.xlsx
Processing: ../RawDataset/rainfall/rain_unguja_kusini.xlsx
Processing: ../RawDataset/rainfall/mjini.xlsx
Processing: ../RawDataset/rainfall/Micheweni.xlsx
Processing: ../RawDataset/rainfall/rain_unguja_kas_b.xlsx
Processing: ../RawDataset/rainfall/rain_unguja_magharibi_a.xlsx
Processing: ../RawDataset/rainfall/rain_unguja_kas_a.xlsx
Processing: ../RawDataset/rainfall/Mkoani.xlsx
Processing: ../RawDataset/rainfall/Chake-Chake.xlsx
Saved combined data to ../RawDataset/rainfall/combined_rainfall_data.csv


In [4]:
folder_path = '../RawDataset/temperatures/max_temp'

all_data = []
for file in os.listdir(folder_path):
    if file.endswith('.xlsx'):
        file_path = os.path.join(folder_path, file)
        print(f"Processing: {file_path}")
        df = clean_rainfall_excel(file_path)
        all_data.append(df)

combined_df = pd.concat(all_data, ignore_index=True)
# Save to CSV
output_csv_path = '../RawDataset/temperatures/max_temp/combined_max_temperature_data.csv'
combined_df.to_csv(output_csv_path, index=False)

print(f"Saved combined data to {output_csv_path}")

Processing: ../RawDataset/temperatures/max_temp/unguja_max_temp.xlsx
Processing: ../RawDataset/temperatures/max_temp/Maximum. t pemba.xlsx
Saved combined data to ../RawDataset/temperatures/max_temp/combined_max_temperature_data.csv


In [5]:
folder_path = '../RawDataset/temperatures/min_temp'

all_data = []
for file in os.listdir(folder_path):
    if file.endswith('.xlsx'):
        file_path = os.path.join(folder_path, file)
        print(f"Processing: {file_path}")
        df = clean_rainfall_excel(file_path)
        all_data.append(df)

combined_df = pd.concat(all_data, ignore_index=True)
# Save to CSV
output_csv_path = '../RawDataset/temperatures/min_temp/combined_min_temperature_data.csv'
combined_df.to_csv(output_csv_path, index=False)

print(f"Saved combined data to {output_csv_path}")

Processing: ../RawDataset/temperatures/min_temp/unguja_min_temp.xlsx
Processing: ../RawDataset/temperatures/min_temp/MINIMUM pemba.xlsx
Saved combined data to ../RawDataset/temperatures/min_temp/combined_min_temperature_data.csv


In [6]:
rainfall_df = pd.read_csv('../RawDataset/rainfall/combined_rainfall_data.csv')
min_temperature_df = pd.read_csv('../RawDataset/temperatures/min_temp/combined_min_temperature_data.csv')
max_temperature_df = pd.read_csv('../RawDataset/temperatures/max_temp/combined_max_temperature_data.csv')

In [7]:
min_temperature_df["Min Temperature"]=min_temperature_df["Rainfall"]
min_temperature_df["Region"]=min_temperature_df["Station"]
min_temperature_df.drop("Station", axis=1, inplace=True)
min_temperature_df.drop("District", axis=1, inplace=True)
min_temperature_df.drop("Rainfall", axis=1, inplace=True)

In [8]:
max_temperature_df["Min Temperature"]=max_temperature_df["Rainfall"]
max_temperature_df["Region"]=max_temperature_df["Station"]
max_temperature_df.drop("Station", axis=1, inplace=True)
max_temperature_df.drop("District", axis=1, inplace=True)
max_temperature_df.drop("Rainfall", axis=1, inplace=True)

In [9]:
min_temperature_df.head()


,Year,Month,Week,Min Temperature,Region
0,2023,Jan,1st week,26.885714,Zanzibar
1,2023,Feb,1st week,25.042857,Zanzibar
2,2023,Mar,1st week,24.771429,Zanzibar
3,2023,Apr,1st week,25.528571,Zanzibar
4,2023,May,1st week,24.657143,Zanzibar


In [10]:
max_temperature_df.head()

,Year,Month,Week,Min Temperature,Region
0,2023,Jan,1st week,32.142857,Zanzibar
1,2023,Feb,1st week,33.357143,Zanzibar
2,2023,Mar,1st week,33.914286,Zanzibar
3,2023,Apr,1st week,31.585714,Zanzibar
4,2023,May,1st week,30.885714,Zanzibar


In [11]:
# Define district-to-region mapping
zanzibar_options = ['WETE', 'MICHEWENI', 'CHAKE-CHAKE', 'MKOANI']
other_options = ['MAGHARIBI  B', 'KUSINI', 'MJINI', 'KASKAZINI B', 'MAGHARIBI  A', 'KASKAZINI A']

# Step 1: Map district to region
def map_district_to_region(district):
    if district.upper() in [d.upper() for d in zanzibar_options]:
        return 'Zanzibar'
    elif district.upper() in [d.upper() for d in other_options]:
        return 'Pemba'
    else:
        return 'Unknown'

rainfall_df['Region'] = rainfall_df['District'].apply(map_district_to_region)

# Step 2: Merge max and min temperature data
temperature_df = pd.merge(
    max_temperature_df.rename(columns={'Min Temperature': 'Max Temperature'}),
    min_temperature_df.rename(columns={'Min Temperature': 'Min Temperature'}),
    on=['Year', 'Month', 'Week', 'Region'],
    how='outer'
)

# Step 3: Merge rainfall and temperature on Year, Month, Week, Region
final_df = pd.merge(
    rainfall_df,
    temperature_df,
    on=['Year', 'Month', 'Week', 'Region'],
    how='left'
)

output_csv_path = '../RawDataset/rainfall_temperature.csv'
final_df.to_csv(output_csv_path, index=False)
# Final result
final_df


,District,Station,Year,Month,Week,Rainfall,Region,Max Temperature,Min Temperature
0,MAGHARIBI B,MAGHARIBI B,2023,Jan,1st week,3.8,Pemba,32.114286,25.200000
1,MAGHARIBI B,MAGHARIBI B,2023,Feb,1st week,0.0,Pemba,31.857143,24.257143
2,MAGHARIBI B,MAGHARIBI B,2023,Mar,1st week,0.0,Pemba,31.957143,24.257143
3,MAGHARIBI B,MAGHARIBI B,2023,Apr,1st week,51.6,Pemba,31.485714,25.314286
4,MAGHARIBI B,MAGHARIBI B,2023,May,1st week,69.6,Pemba,30.142857,25.228571
...,...,...,...,...,...,...,...,...,...
2875,CHAKE-CHAKE,CHAKE-CHAKE,2018,Aug,4th week,0.0,Zanzibar,NaN,NaN
2876,CHAKE-CHAKE,CHAKE-CHAKE,2018,Sep,4th week,38.5,Zanzibar,NaN,NaN
2877,CHAKE-CHAKE,CHAKE-CHAKE,2018,Oct,4th week,61.3,Zanzibar,NaN,NaN
2878,CHAKE-CHAKE,CHAKE-CHAKE,2018,Nov,4th week,3.3,Zanzibar,NaN,NaN


In [12]:
# Function to fit regression and predict for 2018
def predict_temp_for_2018(group_df, temp_col):
    df = group_df.dropna(subset=[temp_col])
    if df['Year'].nunique() < 2:
        return None  # Not enough data to fit a trend
    model = LinearRegression()
    X = df['Year'].values.reshape(-1, 1)
    y = df[temp_col].values
    model.fit(X, y)
    return model.predict([[2018]])[0]

# Iterate over unique District, Month, Week combinations with missing values in 2018
for (district, month, week), group in final_df.groupby(['District', 'Month', 'Week']):
    mask_2018 = (
        (final_df['Year'] == 2018) &
        (final_df['District'] == district) &
        (final_df['Month'] == month) &
        (final_df['Week'] == week)
    )

    if mask_2018.any():
        group_all_years = final_df[
            (final_df['District'] == district) &
            (final_df['Month'] == month) &
            (final_df['Week'] == week) &
            (final_df['Year'] != 2018)
        ]

        pred_max = predict_temp_for_2018(group_all_years, 'Max Temperature')
        pred_min = predict_temp_for_2018(group_all_years, 'Min Temperature')

        if pred_max is not None:
            final_df.loc[mask_2018, 'Max Temperature'] = pred_max
        if pred_min is not None:
            final_df.loc[mask_2018, 'Min Temperature'] = pred_min
final_df
df=final_df


In [13]:
df.head()

,District,Station,Year,Month,Week,Rainfall,Region,Max Temperature,Min Temperature
0,MAGHARIBI B,MAGHARIBI B,2023,Jan,1st week,3.8,Pemba,32.114286,25.200000
1,MAGHARIBI B,MAGHARIBI B,2023,Feb,1st week,0.0,Pemba,31.857143,24.257143
2,MAGHARIBI B,MAGHARIBI B,2023,Mar,1st week,0.0,Pemba,31.957143,24.257143
3,MAGHARIBI B,MAGHARIBI B,2023,Apr,1st week,51.6,Pemba,31.485714,25.314286
4,MAGHARIBI B,MAGHARIBI B,2023,May,1st week,69.6,Pemba,30.142857,25.228571


In [14]:

mosquito = pd.read_excel('../RawDataset/treated mosquito net distribution.xlsx')
coords = pd.read_csv("../RawDataset/sst_coord.csv")

In [15]:
# Step 3: Merge rainfall and temperature on Year, Month, Week, Region
final_df = pd.merge(
    df,
    mosquito,
    on=['Year'],
    how='left'
)

In [16]:
# Step 3: Merge rainfall and temperature on Year, Month, Week, Region
final_df = pd.merge(
    final_df,
    coords,
    on=['District'],
    how='left'
)
df=final_df
df.head()

,District,Station,Year,Month,Week,Rainfall,Region,Max Temperature,Min Temperature,Net distribution,LONGITUDE,LATITUDE
0,MAGHARIBI B,MAGHARIBI B,2023,Jan,1st week,3.8,Pemba,32.114286,25.200000,248000.0,NaN,NaN
1,MAGHARIBI B,MAGHARIBI B,2023,Feb,1st week,0.0,Pemba,31.857143,24.257143,248000.0,NaN,NaN
2,MAGHARIBI B,MAGHARIBI B,2023,Mar,1st week,0.0,Pemba,31.957143,24.257143,248000.0,NaN,NaN
3,MAGHARIBI B,MAGHARIBI B,2023,Apr,1st week,51.6,Pemba,31.485714,25.314286,248000.0,NaN,NaN
4,MAGHARIBI B,MAGHARIBI B,2023,May,1st week,69.6,Pemba,30.142857,25.228571,248000.0,NaN,NaN


In [17]:
# Ensure consistent format
df['Week'] = df['Week'].str.strip()
df['Month'] = df['Month'].str.strip()
# df['Date'] = pd.to_datetime(df['Date'])

# Month name to number mapping (just in case)
month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
df['Month_num'] = df['Month'].map(month_map)

# Week order for sorting
week_order = {
    '1st week': 1,
    '2nd week': 2,
    '3rd week': 3,
    '4th week': 4,
    '5th week': 5
}

# Final output list
daily_rows = []

# Group by Station-Year-Month
grouped = df.groupby(['Station', 'Year', 'Month_num'])

for (station, year, month), group in grouped:
    # group = group.sort_values(by='Week_order')

    # Determine number of days in month
    days_in_month = pd.Timestamp(year=year, month=month, day=1).days_in_month
    start_date = pd.Timestamp(year=year, month=month, day=1)

    # How many weeks in this month?
    n_weeks = len(group)

    # Distribute days across weeks (ensuring sum = days_in_month)
    base_days = days_in_month // n_weeks
    extra_days = days_in_month % n_weeks
    days_per_week = [base_days + (1 if i < extra_days else 0) for i in range(n_weeks)]

    current_day = 0
    for i, (_, row) in enumerate(group.iterrows()):
        num_days = days_per_week[i]

        # Estimate standard deviations
        std_rainfall = row['Rainfall'] * 0.1
        std_temp = 0.5

        # Generate values
        daily_rain = np.clip(
            np.random.normal(loc=row['Rainfall'] / num_days, scale=std_rainfall / num_days, size=num_days),
            0, None
        )
        daily_max_temp = np.random.normal(loc=row['Max Temperature'], scale=std_temp, size=num_days)
        daily_min_temp = np.random.normal(loc=row['Min Temperature'], scale=std_temp, size=num_days)

        for j in range(num_days):
            date = start_date + timedelta(days=current_day)
            current_day += 1
            daily_rows.append({
                'District': row['District'],
                'Station': row['Station'],
                'Region': row['Region'],
                'Net distribution': row.get('Net distribution', np.nan),
                'LONGITUDE': row.get('LONGITUDE', np.nan),
                'LATITUDE': row.get('LATITUDE', np.nan),
                'Date': date.strftime("%Y-%m-%d"),
                'Rainfall': round(daily_rain[j], 2),
                'Max Temperature': round(daily_max_temp[j], 2),
                'Min Temperature': round(daily_min_temp[j], 2)
            })


# Create final daily DataFrame
daily_df = pd.DataFrame(daily_rows)
daily_df.sort_values(by=["Station", "Date"], inplace=True)

# Save to CSV
daily_df.to_csv("normalized_daily_climate_data.csv", index=False)
print(f"Daily data generated successfully. Final shape: {daily_df.shape}")


Daily data generated successfully. Final shape: (21910, 10)


In [18]:
daily_df["District"].unique()

array(['CHAKE-CHAKE', 'KASKAZINI A', 'KASKAZINI B', 'KUSINI',
       'MAGHARIBI  A', 'MAGHARIBI  B', 'MICHEWENI', 'MJINI', 'MKOANI',
       'WETE'], dtype=object)

In [19]:
daily_df

,District,Station,Region,Net distribution,LONGITUDE,LATITUDE,Date,Rainfall,Max Temperature,Min Temperature
0,CHAKE-CHAKE,CHAKE-CHAKE,Zanzibar,NaN,NaN,NaN,2018-01-01,1.94,33.56,25.83
1,CHAKE-CHAKE,CHAKE-CHAKE,Zanzibar,NaN,NaN,NaN,2018-01-02,1.89,33.15,26.53
2,CHAKE-CHAKE,CHAKE-CHAKE,Zanzibar,NaN,NaN,NaN,2018-01-03,1.53,33.29,25.86
3,CHAKE-CHAKE,CHAKE-CHAKE,Zanzibar,NaN,NaN,NaN,2018-01-04,1.53,33.32,26.83
4,CHAKE-CHAKE,CHAKE-CHAKE,Zanzibar,NaN,NaN,NaN,2018-01-05,1.64,33.30,26.22
...,...,...,...,...,...,...,...,...,...,...
21905,WETE,WETE,Zanzibar,NaN,39.92,-5.72,2023-12-27,3.24,30.16,25.71
21906,WETE,WETE,Zanzibar,NaN,39.92,-5.72,2023-12-28,3.41,31.58,25.52
21907,WETE,WETE,Zanzibar,NaN,39.92,-5.72,2023-12-29,3.70,31.27,25.27
21908,WETE,WETE,Zanzibar,NaN,39.92,-5.72,2023-12-30,3.99,30.80,25.83


In [20]:
def read_and_normalize_humidity(file_path):
    # Load monthly humidity data
    df = pd.read_csv(file_path)
    
    # Month name to number
    month_map = {
        'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4,
        'MAY': 5, 'JUN': 6, 'JUL': 7, 'AUG': 8,
        'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12
    }
    df['Month_num'] = df['Month'].map(month_map)
    
    # Create daily humidity values
    daily_rows = []
    
    for index, row in df.iterrows():
        year = int(row['Year'])
        month = int(row['Month_num'])
        region = row['Region']
        monthly_value = float(row['Value'])
    
        # Determine number of days in month
        days_in_month = pd.Timestamp(year, month, 1).days_in_month
    
        # Simulate realistic daily humidity values
        daily_values = np.random.normal(
            loc=monthly_value,        # mean around the monthly average
            scale=1.5,                # small standard deviation for realistic fluctuation
            size=days_in_month
        )
        # Clip values between 0 and 100 (valid humidity range)
        daily_values = np.clip(daily_values, 0, 100)
    
        for day in range(1, days_in_month + 1):
            date = datetime(year, month, day).strftime("%Y-%m-%d")
            daily_rows.append({
                'Region': region,
                'Date': date,
                'Humidity': round(daily_values[day - 1], 1)
            })

    # Create daily DataFrame
    daily_df = pd.DataFrame(daily_rows)
    daily_df.sort_values(by=["Region", "Date"], inplace=True)
    return daily_df


In [21]:
folder_path = '../RawDataset/humidity'
all_data = []
for file in os.listdir(folder_path):
    if file.endswith('.csv'):
        file_path = os.path.join(folder_path, file)
        print(f"Processing: {file_path}")
        df=read_and_normalize_humidity(file_path)
        all_data.append(df)


Processing: ../RawDataset/humidity/humidity_magharibi_a.csv
Processing: ../RawDataset/humidity/pemba_humidity.csv
Processing: ../RawDataset/humidity/humidity_magharibi_b.csv


In [22]:
combined_df = pd.concat(all_data, ignore_index=True)
combined_df["District"] = combined_df["Region"]
combined_df.drop("Region", axis=1, inplace=True)
combined_df

,Date,Humidity,District
0,2018-01-01,73.9,MAGHARIBI A
1,2018-01-02,71.1,MAGHARIBI A
2,2018-01-03,70.7,MAGHARIBI A
3,2018-01-04,69.4,MAGHARIBI A
4,2018-01-05,70.5,MAGHARIBI A
...,...,...,...
6568,2023-12-27,84.4,MAGHARIBI B
6569,2023-12-28,84.9,MAGHARIBI B
6570,2023-12-29,83.7,MAGHARIBI B
6571,2023-12-30,82.9,MAGHARIBI B


In [23]:
# Step 3: Merge rainfall and temperature on Year, Month, Week, Region
final_df = pd.merge(
    daily_df,
    combined_df,
    on=['Date', 'District'],
    how='left'
)

In [24]:
final_df

,District,Station,Region,Net distribution,LONGITUDE,LATITUDE,Date,Rainfall,Max Temperature,Min Temperature,Humidity
0,CHAKE-CHAKE,CHAKE-CHAKE,Zanzibar,NaN,NaN,NaN,2018-01-01,1.94,33.56,25.83,87.9
1,CHAKE-CHAKE,CHAKE-CHAKE,Zanzibar,NaN,NaN,NaN,2018-01-02,1.89,33.15,26.53,90.5
2,CHAKE-CHAKE,CHAKE-CHAKE,Zanzibar,NaN,NaN,NaN,2018-01-03,1.53,33.29,25.86,92.0
3,CHAKE-CHAKE,CHAKE-CHAKE,Zanzibar,NaN,NaN,NaN,2018-01-04,1.53,33.32,26.83,88.9
4,CHAKE-CHAKE,CHAKE-CHAKE,Zanzibar,NaN,NaN,NaN,2018-01-05,1.64,33.30,26.22,91.4
...,...,...,...,...,...,...,...,...,...,...,...
21905,WETE,WETE,Zanzibar,NaN,39.92,-5.72,2023-12-27,3.24,30.16,25.71,NaN
21906,WETE,WETE,Zanzibar,NaN,39.92,-5.72,2023-12-28,3.41,31.58,25.52,NaN
21907,WETE,WETE,Zanzibar,NaN,39.92,-5.72,2023-12-29,3.70,31.27,25.27,NaN
21908,WETE,WETE,Zanzibar,NaN,39.92,-5.72,2023-12-30,3.99,30.80,25.83,NaN


In [25]:
final_df.isna().sum()

District                0
Station                 0
Region                  0
Net distribution    21910
LONGITUDE            6573
LATITUDE             6573
Date                    0
Rainfall              730
Max Temperature         0
Min Temperature         0
Humidity            15337
dtype: int64

In [26]:
# import pandas as pd# Create daily DataFrame
#     daily_df = pd.DataFrame(daily_rows)
#     daily_df.sort_values(by=["Region", "Date"], inplace=True)
    
#     # Save to CSV
#     daily_df.to_csv("daily_normalized_humidity.csv", index=False)
#     print(f"Successfully generated daily humidity data with shape: {daily_df.shape}")

# import numpy as np

# # Step 1: Map weeks to day offsets
# week_map = {
#     '1st week': 0,
#     '2nd week': 7,
#     '3rd week': 14,
#     '4th week': 21
# }

# # Step 2: Generate start date of the week
# def get_week_start_date(row):
#     base = pd.to_datetime(f"{row['Month']} 1 {row['Year']}", format="%b %d %Y")
#     offset = week_map.get(row['Week'], 0)
#     return base + pd.Timedelta(days=offset)

# final_df['Date'] = final_df.apply(get_week_start_date, axis=1)

# # Step 3: Aggregate by Date (mean values in case of duplicates)
# columns_to_interpolate = ['Rainfall', 'Max Temperature', 'Min Temperature']
# daily_base = final_df.groupby('Date')[columns_to_interpolate].mean()

# # Step 4: Reindex with full daily range
# date_range = pd.date_range(start=daily_base.index.min(), end=daily_base.index.max(), freq='D')
# daily_df = daily_base.reindex(date_range)

# # Step 5: Interpolate missing daily values
# daily_df = daily_df.interpolate(method='linear')

# # Optional: Add small daily fluctuations
# np.random.seed(42)
# daily_df['Max Temperature'] += np.random.normal(0, 0.3, size=len(daily_df))
# daily_df['Min Temperature'] += np.random.normal(0, 0.3, size=len(daily_df))

# # Format the index as "dd/mm/yyyy"
# daily_df.index = daily_df.index.strftime("%d/%m/%Y")
# daily_df.reset_index(inplace=True)
# daily_df.rename(columns={'index': 'Date'}, inplace=True)



In [27]:
patients = pd.read_excel("../RawDataset/patient.xlsx")

In [28]:
patients

,Sn,Malaria Case ID,Household Island,Household District,Household Shehia,Date Of Malaria Results,Week,Month,Year,Malaria Positive,...,Sex,Age In Years,AgeGroupLabel,Household Member Type,Facility,Facility District,Facility Type,Household Location - Latitude,Household Location - Longitude,Occupation
0,71368,117360,UNGUJA,MJINI,AMANI,2018-01-16,3,1,2018,True,...,Female,3.0,< 5 yrs,Other Household Member,NaN,NaN,NaN,-6.160946,39.215522,NaN
1,70973,116949,UNGUJA,MJINI,AMANI,2018-01-08,2,1,2018,True,...,Female,6.0,5-14 yrs,Other Household Member,NaN,NaN,NaN,NaN,NaN,NaN
2,70974,116949,UNGUJA,MJINI,AMANI,2018-01-08,2,1,2018,True,...,Female,26.0,15-29 yrs,Other Household Member,NaN,NaN,NaN,NaN,NaN,NaN
3,71369,117360,UNGUJA,MJINI,AMANI,2018-01-16,3,1,2018,True,...,Male,12.0,5-14 yrs,Other Household Member,NaN,NaN,NaN,-6.160946,39.215522,NaN
4,76392,118937,UNGUJA,MJINI,AMANI,2018-05-16,20,5,2018,False,...,Female,2.0,< 5 yrs,NaN,NaN,NaN,NaN,-6.162715,39.216146,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152554,NotInvest,NaN,PEMBA,CHAKECHAKE,ZIWANI,2023-12-20,51,12,2023,True,...,Male,30.0,30-44 yrs,Index case,VITONGOJI COTTAGE HOSPITAL,CHAKECHAKE,NaN,NaN,NaN,NaN
152555,180949,002696Â¹,NaN,NaN,NaN,2023-01-23,4,1,2023,False,...,Female,11.0,5-14 yrs,Other Household Member,NaN,NaN,NaN,NaN,NaN,Student
152556,NotInvest,10970,NaN,MJINI,NaN,2023-11-20,47,11,2023,True,...,Male,28.0,15-29 yrs,Index case,UTAPOA DISPENSARY,MJINI,NaN,NaN,NaN,NaN
152557,213454,16252,UNGUJA,MAGHARIBI A,NaN,2023-12-25,52,12,2023,True,...,Male,30.0,30-44 yrs,Index Case,Unknown,NaN,NaN,NaN,NaN,NaN


In [29]:
patients["Date Of Malaria Results"].nunique()

2188

In [30]:
patients["Household District"].unique()

array(['MJINI', 'KATI', 'KASKAZINI A', 'WETE', 'MAGHARIBI A',
       'KASKAZINI B', 'KUSINI', 'MAGHARIBI B', 'CHAKECHAKE', 'MKOANI',
       'MICHEWENI', nan], dtype=object)

In [31]:
final_df["District"].unique()

array(['CHAKE-CHAKE', 'KASKAZINI A', 'KASKAZINI B', 'KUSINI',
       'MAGHARIBI  A', 'MAGHARIBI  B', 'MICHEWENI', 'MJINI', 'MKOANI',
       'WETE'], dtype=object)

In [32]:
# Rename the column
patients.rename(columns={"Household District": "District"}, inplace=True)

# Mapping of known incorrect spellings to the correct versions
district_corrections = {
    "CHAKECHAKE": "CHAKE-CHAKE",
    "MAGHARIBI A": "MAGHARIBI  A",  # Note: extra space is part of "correct" version
    "MAGHARIBI B": "MAGHARIBI  B",  # Same here
}

# Apply correction (only if value is in the mapping keys)
patients["District"] = patients["District"].replace(district_corrections)

# Optional: strip and standardize formatting (e.g., remove duplicate spaces)
patients["District"] = patients["District"].str.strip().replace(r"\s{2,}", " ", regex=True)

# Optional: show unique values after cleaning
print(patients["District"].unique())


['MJINI' 'KATI' 'KASKAZINI A' 'WETE' 'MAGHARIBI A' 'KASKAZINI B' 'KUSINI'
 'MAGHARIBI B' 'CHAKE-CHAKE' 'MKOANI' 'MICHEWENI' nan]


In [33]:
patients.rename(columns={"Date Of Malaria Results": "Date"}, inplace=True)

In [34]:
final_df['Date'] = pd.to_datetime(final_df['Date'], format="%Y-%m-%d")

In [35]:
merged_final_df = pd.merge(
    patients,
    final_df,
    on=['Date', 'District'],
    how='left'
)

In [36]:
merged_final_df

,Sn,Malaria Case ID,Household Island,District,Household Shehia,Date,Week,Month,Year,Malaria Positive,...,Occupation,Station,Region,Net distribution,LONGITUDE,LATITUDE,Rainfall,Max Temperature,Min Temperature,Humidity
0,71368,117360,UNGUJA,MJINI,AMANI,2018-01-16,3,1,2018,True,...,NaN,MJINI,Pemba,NaN,39.19,-6.16,0.07,31.78,25.83,NaN
1,70973,116949,UNGUJA,MJINI,AMANI,2018-01-08,2,1,2018,True,...,NaN,MJINI,Pemba,NaN,39.19,-6.16,3.75,32.05,24.62,NaN
2,70974,116949,UNGUJA,MJINI,AMANI,2018-01-08,2,1,2018,True,...,NaN,MJINI,Pemba,NaN,39.19,-6.16,3.75,32.05,24.62,NaN
3,71369,117360,UNGUJA,MJINI,AMANI,2018-01-16,3,1,2018,True,...,NaN,MJINI,Pemba,NaN,39.19,-6.16,0.07,31.78,25.83,NaN
4,76392,118937,UNGUJA,MJINI,AMANI,2018-05-16,20,5,2018,False,...,NaN,MJINI,Pemba,NaN,39.19,-6.16,24.75,27.76,24.77,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152554,NotInvest,NaN,PEMBA,CHAKE-CHAKE,ZIWANI,2023-12-20,51,12,2023,True,...,NaN,CHAKE-CHAKE,Zanzibar,NaN,NaN,NaN,3.88,32.24,24.79,89.2
152555,180949,002696Â¹,NaN,NaN,NaN,2023-01-23,4,1,2023,False,...,Student,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152556,NotInvest,10970,NaN,MJINI,NaN,2023-11-20,47,11,2023,True,...,NaN,MJINI,Pemba,NaN,39.19,-6.16,11.96,29.99,25.19,NaN
152557,213454,16252,UNGUJA,MAGHARIBI A,NaN,2023-12-25,52,12,2023,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
df = merged_final_df

In [38]:
df.to_csv("../final_normalized_daily_climate_data.csv", index=False)

In [39]:
df.isna().sum()

Sn                                     0
Malaria Case ID                      592
Household Island                      36
District                              34
Household Shehia                      40
Date                                   0
Week                                   0
Month                                  0
Year                                   0
Malaria Positive                       0
Travel History                    112893
Sex                                 6485
Age In Years                        3108
AgeGroupLabel                          0
Household Member Type              17662
Facility                           96878
Facility District                 138314
Facility Type                     105604
Household Location - Latitude      23146
Household Location - Longitude     23146
Occupation                         42600
Station                            68166
Region                             68166
Net distribution                  152559
LONGITUDE       

In [40]:
df.shape

(152559, 30)

In [41]:
outbreak_df = pd.read_csv("../final_normalized_daily_climate_data.csv")
fumigation = pd.read_excel('../RawDataset/fumigation_area.xlsx')

/tmp/ipykernel_24340/1780149785.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  outbreak_df = pd.read_csv("../final_normalized_daily_climate_data.csv")


In [42]:
outbreak_df.head()

,Sn,Malaria Case ID,Household Island,District,Household Shehia,Date,Week,Month,Year,Malaria Positive,...,Occupation,Station,Region,Net distribution,LONGITUDE,LATITUDE,Rainfall,Max Temperature,Min Temperature,Humidity
0,71368,117360,UNGUJA,MJINI,AMANI,2018-01-16,3,1,2018,True,...,NaN,MJINI,Pemba,NaN,39.19,-6.16,0.07,31.78,25.83,NaN
1,70973,116949,UNGUJA,MJINI,AMANI,2018-01-08,2,1,2018,True,...,NaN,MJINI,Pemba,NaN,39.19,-6.16,3.75,32.05,24.62,NaN
2,70974,116949,UNGUJA,MJINI,AMANI,2018-01-08,2,1,2018,True,...,NaN,MJINI,Pemba,NaN,39.19,-6.16,3.75,32.05,24.62,NaN
3,71369,117360,UNGUJA,MJINI,AMANI,2018-01-16,3,1,2018,True,...,NaN,MJINI,Pemba,NaN,39.19,-6.16,0.07,31.78,25.83,NaN
4,76392,118937,UNGUJA,MJINI,AMANI,2018-05-16,20,5,2018,False,...,NaN,MJINI,Pemba,NaN,39.19,-6.16,24.75,27.76,24.77,NaN


In [43]:
fumigation.head()

,Year,IRS coverage (%),District
0,2018,95.0,NaN
1,2019,95.2,NaN
2,2020,96.9,NaN
3,2021,94.7,NaN
4,2022,97.8,NaN


In [44]:
merged_final_df = pd.merge(
    outbreak_df,
    fumigation,
    on=['Year'],
    how='left'
)

In [45]:
merged_final_df.head()

,Sn,Malaria Case ID,Household Island,District_x,Household Shehia,Date,Week,Month,Year,Malaria Positive,...,Region,Net distribution,LONGITUDE,LATITUDE,Rainfall,Max Temperature,Min Temperature,Humidity,IRS coverage (%),District_y
0,71368,117360,UNGUJA,MJINI,AMANI,2018-01-16,3,1,2018,True,...,Pemba,NaN,39.19,-6.16,0.07,31.78,25.83,NaN,95.0,NaN
1,70973,116949,UNGUJA,MJINI,AMANI,2018-01-08,2,1,2018,True,...,Pemba,NaN,39.19,-6.16,3.75,32.05,24.62,NaN,95.0,NaN
2,70974,116949,UNGUJA,MJINI,AMANI,2018-01-08,2,1,2018,True,...,Pemba,NaN,39.19,-6.16,3.75,32.05,24.62,NaN,95.0,NaN
3,71369,117360,UNGUJA,MJINI,AMANI,2018-01-16,3,1,2018,True,...,Pemba,NaN,39.19,-6.16,0.07,31.78,25.83,NaN,95.0,NaN
4,76392,118937,UNGUJA,MJINI,AMANI,2018-05-16,20,5,2018,False,...,Pemba,NaN,39.19,-6.16,24.75,27.76,24.77,NaN,95.0,NaN


In [46]:
merged_final_df.isna().sum()


Sn                                     0
Malaria Case ID                      592
Household Island                      36
District_x                            34
Household Shehia                      40
Date                                   0
Week                                   0
Month                                  0
Year                                   0
Malaria Positive                       0
Travel History                    112893
Sex                                 6485
Age In Years                        3108
AgeGroupLabel                          0
Household Member Type              17662
Facility                           96878
Facility District                 138314
Facility Type                     105604
Household Location - Latitude      23146
Household Location - Longitude     23146
Occupation                         42600
Station                            68166
Region                             68166
Net distribution                  152559
LONGITUDE       

In [47]:
def clean_malaria_data(df):
    # Make a copy of the original dataframe
    cleaned_df = df.copy()
    
    # 1. Ensure datetime is properly formatted and set as index for time operations
    cleaned_df['Date'] = pd.to_datetime(cleaned_df['Date'])
    cleaned_df.sort_values(by=['Household Island', 'Date'], inplace=True)
    
    # Weather variables to impute
    weather_vars = ['Rainfall', 'Max Temperature', 'Min Temperature', 'Humidity']
    location_vars = ['Household Location - Latitude', 'Household Location - Longitude']
    
    # 2. Geospatial imputation for location data first
    if cleaned_df[location_vars].isna().any().any():
        # Use district centroids for missing locations
        district_locations = cleaned_df.dropna(subset=location_vars).groupby('District_x')[
            ['Household Location - Latitude', 'Household Location - Longitude']].median()
        
        for idx, row in cleaned_df[cleaned_df[location_vars].isna().any(axis=1)].iterrows():
            district = row['District_x']
            if district in district_locations.index:
                cleaned_df.at[idx, 'Household Location - Latitude'] = district_locations.loc[district, 'Household Location - Latitude']
                cleaned_df.at[idx, 'Household Location - Longitude'] = district_locations.loc[district, 'Household Location - Longitude']
    
    # 3. Time-series imputation for weather variables by location
    for island in cleaned_df['Household Island'].unique():
        island_mask = cleaned_df['Household Island'] == island
        island_df = cleaned_df.loc[island_mask].copy()
        
        # Create a temporary datetime index for interpolation
        temp_df = island_df.set_index('Date').sort_index()
        
        for var in weather_vars:
            # First try linear interpolation (numeric)
            temp_df[var] = temp_df[var].interpolate(
                method='linear',
                limit_direction='both'
            )
            
            # If still missing, use seasonal decomposition if enough data
            if temp_df[var].isna().any():
                try:
                    # Need at least 2 full periods (24 months) for seasonal decomposition
                    if len(temp_df[var].dropna()) >= 24:
                        result = seasonal_decompose(
                            temp_df[var].dropna(),
                            period=12,  # monthly seasonality
                            model='additive',
                            extrapolate_trend='freq'
                        )
                        temp_df[var] = result.trend + result.seasonal + result.resid
                    # Fill any remaining NAs with rolling mean
                    temp_df[var] = temp_df[var].fillna(
                        temp_df[var].rolling(7, min_periods=1).mean()
                    )
                except:
                    # Fallback to simple forward/backward fill
                    temp_df[var] = temp_df[var].fillna(method='ffill').fillna(method='bfill')
            
            # Update the cleaned_df with imputed values
            cleaned_df.loc[island_mask, var] = temp_df[var].values
    
    # 4. For remaining weather missingness, use spatial KNN imputation
    if cleaned_df[weather_vars].isna().any().any():
        # Prepare spatial features
        spatial_features = cleaned_df[['Household Location - Latitude', 'Household Location - Longitude']].copy()
        
        # Add temporal feature (days since first observation)
        min_date = cleaned_df['Date'].min()
        spatial_features['Days'] = (cleaned_df['Date'] - min_date).dt.days
        
        # KNN imputer for weather variables
        knn_imputer = KNNImputer(n_neighbors=5, weights='distance')
        weather_imputed = knn_imputer.fit_transform(cleaned_df[weather_vars])
        cleaned_df[weather_vars] = weather_imputed
    
    # 5. Handle other variables with MICE imputation
    other_vars = ['Travel History', 'Sex', 'Age In Years', 'Occupation', 'Household Member Type']
    
    # Convert categorical variables to numeric for imputation
    cat_vars = ['Travel History', 'Sex', 'Occupation', 'Household Member Type']
    for var in cat_vars:
        cleaned_df[var+'_encoded'] = pd.factorize(cleaned_df[var])[0]
    
    # Use IterativeImputer (MICE implementation)
    imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=100, random_state=42),
        max_iter=10,
        random_state=42
    )
    
    # Prepare data for imputation
    impute_data = cleaned_df[[v+'_encoded' if v in cat_vars else v for v in other_vars]]
    imputed_data = imputer.fit_transform(impute_data)
    
    # Put back imputed values
    for i, var in enumerate(other_vars):
        if var in cat_vars:
            # Round and convert back to categorical
            imputed_cats = np.round(imputed_data[:, i]).astype(int)
            cleaned_df[var] = pd.Categorical.from_codes(
                imputed_cats,
                categories=cleaned_df[var].dropna().unique()
            )
            cleaned_df.drop(columns=[var+'_encoded'], inplace=True)
        else:
            cleaned_df[var] = imputed_data[:, i]
    
    # 6. Drop columns with excessive missingness that can't be reasonably imputed
    cols_to_drop = ['Net distribution', 'Facility District', 'Facility Type', 'Facility']
    cleaned_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    
    # 7. Final cleanup - drop any remaining rows with critical missing values
    critical_vars = ['Malaria Positive', 'AgeGroupLabel', 'District_x', 'Household Shehia']
    cleaned_df.dropna(subset=critical_vars, inplace=True)
    
    return cleaned_df

# Apply the cleaning function to your dataframe
cleaned_data = clean_malaria_data(merged_final_df)

In [48]:
cleaned_data.head()

,Sn,Malaria Case ID,Household Island,District_x,Household Shehia,Date,Week,Month,Year,Malaria Positive,...,Station,Region,LONGITUDE,LATITUDE,Rainfall,Max Temperature,Min Temperature,Humidity,IRS coverage (%),District_y
6661,70576,117174,PEMBA,MICHEWENI,KONDE,2018-01-01,1,1,2018,True,...,MICHEWENI,Zanzibar,40.04,-4.98,0.79,33.43,25.80,90.50,95.0,NaN
15882,70579,117365,PEMBA,WETE,UKUNJWI,2018-01-01,1,1,2018,True,...,WETE,Zanzibar,39.92,-5.72,8.66,34.93,25.10,90.50,95.0,NaN
15892,70581,117658,PEMBA,WETE,UKUNJWI,2018-01-01,1,1,2018,True,...,WETE,Zanzibar,39.92,-5.72,8.66,34.93,25.10,90.50,95.0,NaN
979,70676,117189,PEMBA,CHAKE-CHAKE,CHACHANI,2018-01-02,1,1,2018,True,...,CHAKE-CHAKE,Zanzibar,NaN,NaN,1.89,33.15,26.53,90.50,95.0,NaN
2970,70679,117206,PEMBA,WETE,GANDO,2018-01-02,1,1,2018,True,...,WETE,Zanzibar,39.92,-5.72,7.75,33.17,27.21,90.55,95.0,NaN


In [49]:
cleaned_data.isna().sum()

Sn                                     0
Malaria Case ID                      592
Household Island                       0
District_x                             0
Household Shehia                       0
Date                                   0
Week                                   0
Month                                  0
Year                                   0
Malaria Positive                       0
Travel History                    112862
Sex                                 6460
Age In Years                           0
AgeGroupLabel                          0
Household Member Type              17659
Household Location - Latitude          0
Household Location - Longitude         0
Occupation                         42566
Station                            68128
Region                             68128
LONGITUDE                          72323
LATITUDE                           72323
Rainfall                               0
Max Temperature                        0
Min Temperature 

In [50]:
cleaned_data.to_csv("../final_combined_normalized_daily_climate_data.csv", index=False)

In [54]:
cleaned_dataa=pd.read_csv("../final_combined_normalized_daily_climate_data.csv")

/tmp/ipykernel_24340/2429637658.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  cleaned_dataa=pd.read_csv("../final_combined_normalized_daily_climate_data.csv")


In [55]:
cleaned_data.drop("LONGITUDE", axis=1, inplace=True)
cleaned_data.drop("LATITUDE", axis=1, inplace=True)
cleaned_data.drop("Region", axis=1, inplace=True)
cleaned_data.drop("Station", axis=1, inplace=True)
cleaned_data.drop("District_y", axis=1, inplace=True)

In [56]:
cleaned_data.shape

(152519, 23)

In [57]:
cleaned_data.isna().sum()

Sn                                     0
Malaria Case ID                      592
Household Island                       0
District_x                             0
Household Shehia                       0
Date                                   0
Week                                   0
Month                                  0
Year                                   0
Malaria Positive                       0
Travel History                    112862
Sex                                 6460
Age In Years                           0
AgeGroupLabel                          0
Household Member Type              17659
Household Location - Latitude          0
Household Location - Longitude         0
Occupation                         42566
Rainfall                               0
Max Temperature                        0
Min Temperature                        0
Humidity                               0
IRS coverage (%)                       0
dtype: int64

In [58]:
cleaned_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152519 entries, 0 to 152518
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Sn                              152519 non-null  object 
 1   Malaria Case ID                 151927 non-null  object 
 2   Household Island                152519 non-null  object 
 3   District_x                      152519 non-null  object 
 4   Household Shehia                152519 non-null  object 
 5   Date                            152519 non-null  object 
 6   Week                            152519 non-null  int64  
 7   Month                           152519 non-null  int64  
 8   Year                            152519 non-null  int64  
 9   Malaria Positive                152519 non-null  bool   
 10  Travel History                  39657 non-null   object 
 11  Sex                             146059 non-null  object 
 12  Age In Years    

In [59]:
cleaned_data.nunique()

Sn                                144482
Malaria Case ID                    54607
Household Island                       2
District_x                            11
Household Shehia                     451
Date                                2188
Week                                  53
Month                                 12
Year                                   6
Malaria Positive                       2
Travel History                         2
Sex                                    2
Age In Years                         239
AgeGroupLabel                          5
Household Member Type                  4
Household Location - Latitude      34060
Household Location - Longitude     32945
Occupation                            11
Rainfall                           26497
Max Temperature                    25808
Min Temperature                    23631
Humidity                           51675
IRS coverage (%)                       6
dtype: int64

In [60]:
def final_cleaning(cleaned_df):
    # Make a copy to avoid SettingWithCopyWarning
    df = cleaned_df.copy()
    
    # 1. Handle Travel History
    if 'Travel History' in df.columns:
        if df['Travel History'].isna().any():
            df['Travel History'] = df['Travel History'].cat.add_categories(['Unknown']).fillna('Unknown')
    
    # 2. Impute Sex - safer implementation
    if 'Sex' in df.columns and df['Sex'].isna().any():
        # Get mode by household first
        sex_modes = df.groupby('Household Island')['Sex'].apply(lambda x: x.mode()[0] if not x.mode().empty else pd.NA)
        
        # Fill household-level modes
        na_mask = df['Sex'].isna()
        df.loc[na_mask, 'Sex'] = df.loc[na_mask, 'Household Island'].map(sex_modes)
        
        # Fill remaining with overall mode
        if df['Sex'].isna().any():
            overall_mode = df['Sex'].mode()[0]
            df['Sex'] = df['Sex'].fillna(overall_mode)
    
    # 3. Household Member Type - fixed implementation with proper categorical handling
    if all(col in df.columns for col in ['Age In Years', 'Household Member Type', 'Sex']):
        # Add new categories if they don't exist
        new_categories = ['Child', 'Female Adult', 'Male Adult', 'Unknown']
        current_categories = df['Household Member Type'].cat.categories
        missing_categories = [cat for cat in new_categories if cat not in current_categories]
        
        if missing_categories:
            df['Household Member Type'] = df['Household Member Type'].cat.add_categories(missing_categories)
        
        # Initialize with current values
        member_type = df['Household Member Type'].copy()
        
        # Create masks for each category
        child_mask = (df['Age In Years'] < 18) & member_type.isna()
        female_adult_mask = (df['Age In Years'] >= 18) & (df['Sex'] == 'Female') & member_type.isna()
        male_adult_mask = (df['Age In Years'] >= 18) & (df['Sex'] == 'Male') & member_type.isna()
        
        # Apply imputations
        member_type = member_type.astype('object')  # Temporarily convert to object to allow new categories
        member_type[child_mask] = 'Child'
        member_type[female_adult_mask] = 'Female Adult'
        member_type[male_adult_mask] = 'Male Adult'
        member_type = member_type.astype('category')  # Convert back to category
        
        # Fill remaining with mode
        if member_type.isna().any():
            overall_mode = member_type.mode()[0]
            member_type = member_type.fillna(overall_mode)
        
        df['Household Member Type'] = member_type
    
    # 4. Occupation - safer implementation
    if 'Occupation' in df.columns and df['Occupation'].isna().any():
        # Initialize with current values
        occupation = df['Occupation'].copy()
        
        # Child rule
        child_mask = (df['Age In Years'] < 18) & occupation.isna()
        occupation = occupation.astype('object')  # Temporarily convert to object
        occupation[child_mask] = 'Student'
        
        # Adult rules
        if 'Sex' in df.columns:
            adult_mask = (df['Age In Years'] >= 18) & occupation.isna()
            female_adult_mask = adult_mask & (df['Sex'] == 'Female')
            male_adult_mask = adult_mask & (df['Sex'] == 'Male')
            
            occupation[female_adult_mask] = 'Household'
            occupation[male_adult_mask] = 'Farmer'
        
        # Fill remaining with mode
        if occupation.isna().any():
            overall_mode = occupation.mode()[0]
            occupation = occupation.fillna(overall_mode)
        
        df['Occupation'] = occupation.astype('category')  # Convert back to category
    
    # 5. Malaria Case ID
    if 'Malaria Case ID' in df.columns and df['Malaria Case ID'].isna().any():
        existing_ids = df['Malaria Case ID'].dropna()
        if not existing_ids.empty:
            max_id = existing_ids.str.extract(r'(\d+)').astype(int).max().values[0]
        else:
            max_id = 0
        missing_mask = df['Malaria Case ID'].isna()
        df.loc[missing_mask, 'Malaria Case ID'] = [f'CASE_{x}' for x in range(max_id + 1, max_id + 1 + missing_mask.sum())]
    
    return df

# Apply final cleaning
fully_cleaned_data = final_cleaning(cleaned_data)

AttributeError: Can only use .cat accessor with a 'category' dtype

In [45]:
fully_cleaned_data.isna().sum()

Malaria Case ID                   0
Household Island                  0
District_x                        0
Household Shehia                  0
Date                              0
Week                              0
Month                             0
Year                              0
Malaria Positive                  0
Travel History                    0
Sex                               0
Age In Years                      0
AgeGroupLabel                     0
Household Member Type             0
Household Location - Latitude     0
Household Location - Longitude    0
Occupation                        0
Rainfall                          0
Max Temperature                   0
Min Temperature                   0
Humidity                          0
IRS coverage (%)                  0
dtype: int64

In [48]:
fully_cleaned_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 152519 entries, 6661 to 152499
Data columns (total 22 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   Malaria Case ID                 152519 non-null  object        
 1   Household Island                152519 non-null  object        
 2   District_x                      152519 non-null  object        
 3   Household Shehia                152519 non-null  object        
 4   Date                            152519 non-null  datetime64[ns]
 5   Week                            152519 non-null  int64         
 6   Month                           152519 non-null  int64         
 7   Year                            152519 non-null  int64         
 8   Malaria Positive                152519 non-null  bool          
 9   Travel History                  152519 non-null  category      
 10  Sex                             152519 non-null  category 

In [47]:
fully_cleaned_data.shape

(152519, 22)

In [ ]:
fully_cleaned_data.to_csv("../final_combined_cleaned_normalized_daily_climate_data.csv", index=False)

In [55]:
class MalariaOutbreakPredictor:
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        self.model = None
        self.feature_columns = None
        self.categorical_features = None
        self.numeric_features = None
        
    def prepare_data(self, df):
        """Prepare features and target variable"""
        # Create target variable
        df['Outbreak'] = (df['Malaria Positive'].astype(int) & 
                         (df.groupby(['District_x', 'Week'])['Malaria Positive']
                         .transform('sum') > df.groupby('District_x')['Malaria Positive']
                         .transform('mean')*2)).astype(int)
        
        # Select features
        features = [
            'Week', 'Month', 'Year', 'Rainfall', 'Max Temperature', 
            'Min Temperature', 'Humidity', 'IRS coverage (%)', 'Age In Years',
            'Sex', 'Travel History', 'Household Member Type', 'Occupation',
            'District_x', 'Household Shehia'
        ]
        
        return df[features], df['Outbreak']
    
    def train_model(self, X, y):
        """Train the prediction model with threshold tuning"""
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y)
        
        # Identify feature types
        self.numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
        self.categorical_features = X_train.select_dtypes(include=['category', 'object']).columns.tolist()
        self.feature_columns = self.numeric_features + self.categorical_features
        
        # Convert categoricals to strings (avoids sparse matrix issues)
        for col in self.categorical_features:
            X_train[col] = X_train[col].astype(str)
            X_test[col] = X_test[col].astype(str)
        
        # Create preprocessor
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ])
        
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, self.numeric_features),
                ('cat', categorical_transformer, self.categorical_features)
            ])
        
        # Create model pipeline
        self.model = make_pipeline(
            preprocessor,
            HistGradientBoostingClassifier(
                random_state=42,
                categorical_features=[i for i, col in enumerate(self.feature_columns) 
                                    if col in self.categorical_features]
            )
        )
        
        # Fit model
        self.model.fit(X_train, y_train)
        
        # Tune threshold
        self.tune_threshold(X_test, y_test)
        
        return self.model
    
    def tune_threshold(self, X, y_true):
        """Find optimal threshold based on precision-recall tradeoff"""
        y_proba = self.model.predict_proba(X)[:, 1]
        
        thresholds = np.linspace(0.1, 0.9, 50)
        best_score = -1
        best_threshold = 0.5
        
        for threshold in thresholds:
            y_pred = (y_proba >= threshold).astype(int)
            report = classification_report(y_true, y_pred, output_dict=True)
            f2_score = report['1']['f1-score']
            if f2_score > best_score:
                best_score = f2_score
                best_threshold = threshold
                
        self.threshold = best_threshold
        print(f"Optimal threshold set to: {self.threshold:.3f}")
    
    def predict(self, X):
        """Make predictions with the tuned threshold"""
        if self.model is None:
            raise ValueError("Model not trained yet. Call train_model first.")
            
        # Ensure we have all expected columns (fill missing with NaN)
        X = X.reindex(columns=self.feature_columns, fill_value=np.nan)
        
        # Convert categoricals to strings
        for col in self.categorical_features:
            if col in X.columns:
                X[col] = X[col].astype(str)
        
        proba = self.model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)
    
    def save_model(self, path):
        """Save the trained model"""
        joblib.dump({
            'model': self.model,
            'threshold': self.threshold,
            'features': self.feature_columns,
            'cat_features': self.categorical_features,
            'num_features': self.numeric_features
        }, path)
    
    @classmethod
    def load_model(cls, path):
        """Load a trained model"""
        data = joblib.load(path)
        predictor = cls(threshold=data['threshold'])
        predictor.model = data['model']
        predictor.feature_columns = data['features']
        predictor.categorical_features = data['cat_features']
        predictor.numeric_features = data['num_features']
        return predictor

In [56]:
# Initialize predictor
predictor = MalariaOutbreakPredictor()

# Prepare data
X, y = predictor.prepare_data(fully_cleaned_data)

# Train model
model = predictor.train_model(X, y)

# Save model
predictor.save_model('malaria_outbreak_predictor.joblib')

ValueError: Categorical feature at index 5 is expected to have a cardinality <= 255 but actually has a cardinality of 445.